In [20]:
import numpy as np
import pandas as pd
import re

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

In [22]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error, median_absolute_error

In [23]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("XGBoost_Regressor_HPO")
mlflow.set_tracking_uri("http://localhost:5000")

2025/07/02 09:06:15 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Regressor_HPO' does not exist. Creating a new experiment.


In [24]:
import json
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from optuna.samplers import TPESampler
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, HalvingRandomSearchCV

In [25]:
param_list = {}

In [26]:
# Define a function to calculate all metrics and format a report
def calculate_metrics(y, y_pred, y_val = None, y_pred_val = None, verbose=True):
    # Calculate train metrics
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    evs = explained_variance_score(y, y_pred)
    max_err = max_error(y, y_pred)
    medae = median_absolute_error(y, y_pred)
    metrics = {
        "mse": mse,
        "r2": r2,
        "mae": mae,
        "evs": evs,
        "max_err": max_err,
        "medae": medae
    }

    # If validation data is provided, calculate validation metrics
    if y_val is not None and y_pred_val is not None:
        # Calculate validation metrics
        mse_val = mean_squared_error(y_val, y_pred_val)
        r2_val = r2_score(y_val, y_pred_val)
        mae_val = mean_absolute_error(y_val, y_pred_val)
        evs_val = explained_variance_score(y_val, y_pred_val)
        max_err_val = max_error(y_val, y_pred_val)
        medae_val = median_absolute_error(y_val, y_pred_val)
        metrics.update({
            "mse_val": mse_val,
            "r2_val": r2_val,
            "mae_val": mae_val,
            "evs_val": evs_val,
            "max_err_val": max_err_val,
            "medae_val": medae_val
        })

        # Create a table for formatting metrics
        if verbose:
            print("Metric       |    Train     |  Validation")
            print("-------------|--------------|---------------")
            print(f"MSE          |   {mse:.2e}   |   {mse_val:.2e}")
            print(f"R²           |   {r2:.4f}     |   {r2_val:.4f}")
            print(f"MAE          |   {mae:.2e}   |   {mae_val:.2e}")
            print(f"EVS          |   {evs:.4f}     |   {evs_val:.4f}")
            print(f"Max Error    |   {max_err:.2e}   |   {max_err_val:.2e}")
            print(f"Median AE    |   {medae:.2e}   |   {medae_val:.2e}")
            print()
            
    elif verbose:
        print(f"MSE          |   {mse:.2e}   ")
        print(f"R²           |   {r2:.4f}     ")
        print(f"MAE          |   {mae:.2e}   ")
        print(f"EVS          |   {evs:.4f}     ")
        print(f"Max Error    |   {max_err:.2e}   ")
        print(f"Median AE    |   {medae:.2e}   ")
        print()
    
    # Return metrics in a dictionary for reporting
    return metrics

## Preparacion de datos

In [27]:
df_train = pd.read_csv('../data/train/transformed_train.csv')
df_train.info()

# Variable target
y_train = df_train["Precio"]
X_train = df_train.drop(columns=["Precio"])

df_val = pd.read_csv('../data/train/transformed_val.csv')
X_val = df_val.drop(columns=["Precio"])
y_val = df_val["Precio"]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11682 entries, 0 to 11681
Data columns (total 81 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Marca                              11682 non-null  float64
 1   Modelo                             11682 non-null  float64
 2   Versión                            11682 non-null  float64
 3   Motor                              11664 non-null  float64
 4   Turbo                              11682 non-null  int64  
 5   cv                                 6145 non-null   float64
 6   Puertas                            11682 non-null  float64
 7   Kilómetros                         11682 non-null  float64
 8   Precio                             11682 non-null  float64
 9   Año                                11682 non-null  float64
 10  desc_new_svd_1                     11682 non-null  float64
 11  desc_new_svd_2                     11682 non-null  flo

## First Experiment

Ajuste de hiperparametros

In [28]:
# Espacio de hiperparámetros refinado
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2, 0.5],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [1, 1.5, 2]
}

# Modelo base
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

# RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Entrenar
search.fit(X_train, y_train)
best_model = search.best_estimator_
best_params = search.best_params_

# Predicciones
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["RandomizedSearchCV"]=best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   2.84e+13   |   4.49e+13
R²           |   0.9438     |   0.9006
MAE          |   2.97e+06   |   3.36e+06
EVS          |   0.9438     |   0.9007
Max Error    |   1.93e+08   |   1.81e+08
Median AE    |   1.97e+06   |   2.00e+06

Best parameters found:
{
    "subsample": 0.6,
    "reg_lambda": 1.5,
    "reg_alpha": 0.1,
    "n_estimators": 400,
    "max_depth": 3,
    "learning_rate": 0.1,
    "gamma": 0.5,
    "colsample_bytree": 1.0
}


In [29]:
# Espacio de hiperparámetros refinado
param_dist = {
    'learning_rate': [0.05, 0.075, 0.1, 0.125],
    'n_estimators': [250, 300, 350, 400],
    'max_depth': [5, 6, 7],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.7],
    'gamma': [0, 0.1, 0.3],
    'reg_alpha': [0, 0.005, 0.01, 0.05],
    'reg_lambda': [0.8, 1, 1.2, 1.5],
    'min_child_weight': [1, 3, 5]
}

# Modelo base
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

# RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Entrenar
search.fit(X_train, y_train)
best_model = search.best_estimator_
best_params = search.best_params_

# Predicciones
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

# Crear reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["RandomizedSearchCV_2"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.51e+13   |   3.47e+13
R²           |   0.9702     |   0.9232
MAE          |   2.00e+06   |   2.85e+06
EVS          |   0.9702     |   0.9232
Max Error    |   2.19e+08   |   1.75e+08
Median AE    |   1.34e+06   |   1.61e+06

Best parameters found:
{
    "subsample": 0.8,
    "reg_lambda": 0.8,
    "reg_alpha": 0,
    "n_estimators": 350,
    "min_child_weight": 3,
    "max_depth": 5,
    "learning_rate": 0.1,
    "gamma": 0.3,
    "colsample_bytree": 0.5
}


In [30]:
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.2, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 300, 600),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.001, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 2.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 0.3),
        'tree_method': 'hist',
        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide'])
    }

    model = XGBRegressor(**params, objective='reg:squarederror', random_state=42)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_root_mean_squared_error').mean()
    return -score  # minimizar RMSE → negativo

# Optimización
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="minimize", sampler=TPESampler())
study.optimize(objective, n_trials=100)

# Entrenamiento final con mejores hiperparámetros
best_params = study.best_params
best_model = XGBRegressor(**best_params, objective='reg:squarederror', random_state=42)
best_model.fit(X_train, y_train)

# Evaluación
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["Optuna_refined"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))


Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.72e+13   |   3.30e+13
R²           |   0.9661     |   0.9270
MAE          |   2.09e+06   |   2.87e+06
EVS          |   0.9661     |   0.9270
Max Error    |   2.23e+08   |   1.49e+08
Median AE    |   1.40e+06   |   1.70e+06

Best parameters found:
{
    "max_depth": 5,
    "learning_rate": 0.08781478142400266,
    "n_estimators": 355,
    "subsample": 0.9607649093079691,
    "colsample_bytree": 0.6095411770550445,
    "reg_alpha": 0.05818268324015919,
    "reg_lambda": 1.0709258135064956,
    "min_child_weight": 8,
    "gamma": 0.0332403507510049,
    "grow_policy": "lossguide"
}


In [31]:
halving_search = HalvingRandomSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    factor=3,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)
halving_search.fit(X_train, y_train)
best_model = halving_search.best_estimator_
best_params = halving_search.best_params_

# Predicciones
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)
# Crear reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["HalvingRandomSearchCV"] = best_params


Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.77e+13   |   3.87e+13
R²           |   0.9651     |   0.9144
MAE          |   2.24e+06   |   2.97e+06
EVS          |   0.9651     |   0.9144
Max Error    |   1.98e+08   |   1.74e+08
Median AE    |   1.52e+06   |   1.70e+06



In [32]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer

# Definir el espacio de búsqueda bayesiana
search_spaces = {
    'n_estimators': Integer(300, 500),
    'max_depth': Integer(5, 7),
    'learning_rate': Real(0.05, 0.15, prior='log-uniform'),
    'subsample': Real(0.8, 1.0),
    'colsample_bytree': Real(0.5, 0.7),
    'gamma': Real(0.0, 0.2),
    'reg_alpha': Real(0.005, 0.05, prior='log-uniform'),
    'reg_lambda': Real(0.8, 1.5)
}

# Configurar el modelo base
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

# Configurar la búsqueda bayesiana
opt = BayesSearchCV(
    estimator=xgb,
    search_spaces=search_spaces,
    n_iter=40,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

# Entrenar
opt.fit(X_train, y_train)
best_model = opt.best_estimator_
best_params = opt.best_params_

# Evaluación
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["BayesSearchCV"] = best_params

print("Best Bayes parameters found:")
print(json.dumps(best_params, indent=4))


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fi

In [34]:
input_example = X_train.iloc[0:1]
artifact_csv_path = "data/train/cleaned_train.csv"  # ajusta la ruta si es necesario

for name, params in param_list.items():
    model = XGBRegressor(objective='reg:squarederror', random_state=42, **params)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    metrics = calculate_metrics(y_val, y_pred_val, verbose=False)

    run_name = f"XGBoost_{name}"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        signature = infer_signature(input_example, model.predict(input_example))
        mlflow.xgboost.log_model(model, artifact_path="model", input_example=input_example, signature=signature)

/home/elianaostro/miniconda3/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/07/02 10:07:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/home/elianaostro/miniconda3/lib/python3.12/site-packages/xgboost/sklearn.py:1028: UserWarning: [10:07:16] WARNING: /worksp

🏃 View run XGBoost_Optuna_refined at: http://localhost:5000/#/experiments/504106521660686195/runs/0b09992841e84f6da38c3c6a66797294
🧪 View experiment at: http://localhost:5000/#/experiments/504106521660686195


In [35]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
import json

# Definir espacio exhaustivo
param_grid = {
    'n_estimators': [300, 400, 500],
    'max_depth': [5, 6, 7],
    'learning_rate': [0.05, 0.1, 0.15],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.7],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0.005, 0.01, 0.05],
    'reg_lambda': [1.0, 1.2, 1.5],
    'min_child_weight': [1, 3, 5],
    'tree_method': ['hist'],
    'grow_policy': ['depthwise', 'lossguide']
}

# Definir modelo base
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

# RandomizedSearchCV
grid_search = RandomizedSearchCV(
    estimator=xgb,
    n_iter=300,  # número de combinaciones a probar
    param_distributions=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1,
    return_train_score=True
)

# Entrenamiento
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 300 candidates, totalling 900 fits


RandomizedSearchCV(cv=3,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                   param_distributions={'colsample_bytree': [0.5, 0.6, 0.7],
                                        'gamma': [0, 0.1, 0.2],
                                        'grow_policy': ['depthwise',
                                                        'lossguide'],
                                        'learning_rate': [0.05, 0.1, 0.15],
                                        'max_depth': [5, 6, 7],
                                        'min_child_weight': [1, 3, 5],
                                        'n_estimators': [300, 400, 500],
                                        'reg_alpha': [0.005, 0.01, 0.05],
                                        'reg_lambda': [1.0, 1.2, 1.5],
                                        'subsample': [0.8, 0.9, 1.0],
                                        'tree_method': ['hist']},
                   return_train_score=True,
                   scoring='neg_root_mean_squared_error', verbose=1)

In [36]:
# Mejor modelo
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_

# Evaluar
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

train_metrics = calculate_metrics(y_train, y_pred_train)
val_metrics = calculate_metrics(y_val, y_pred_val)

# Registrar mejor modelo en MLflow
with mlflow.start_run(run_name="xgb_best_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({f"train_{k}": v for k, v in train_metrics.items()})
    mlflow.log_metrics({f"val_{k}": v for k, v in val_metrics.items()})
    mlflow.sklearn.log_model(best_model, artifact_path="model")

# También lo podés guardar en tu variable param_list
param_list["GridSearchCV_full"] = best_params

print("Best GridSearchCV parameters found:")
print(json.dumps(best_params, indent=4))


2025/07/02 10:21:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MSE          |   1.77e+13   
R²           |   0.9650     
MAE          |   2.14e+06   
EVS          |   0.9650     
Max Error    |   2.27e+08   
Median AE    |   1.41e+06   

MSE          |   3.40e+13   
R²           |   0.9248     
MAE          |   2.84e+06   
EVS          |   0.9248     
Max Error    |   1.67e+08   
Median AE    |   1.67e+06   



2025/07/02 10:21:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run xgb_best_model at: http://localhost:5000/#/experiments/504106521660686195/runs/b85e97e64864423080ae5b61d0d1e334
🧪 View experiment at: http://localhost:5000/#/experiments/504106521660686195
Best GridSearchCV parameters found:
{
    "tree_method": "hist",
    "subsample": 1.0,
    "reg_lambda": 1.0,
    "reg_alpha": 0.05,
    "n_estimators": 500,
    "min_child_weight": 5,
    "max_depth": 5,
    "learning_rate": 0.05,
    "grow_policy": "lossguide",
    "gamma": 0,
    "colsample_bytree": 0.6
}
